## Setup and load data

RQ2 on the stratified folds: does the shared encoder help or hurt compared to training one model per target. I use the same seed, stratified folds and training procedure as the authoritative run in Notebook 8, so the only difference is single-task versus multi-task.

In [1]:
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False

device="cuda" if torch.cuda.is_available() else "cpu"
print("device:",device)
print("gpu:",torch.cuda.get_device_name(0) if device=="cuda" else "none")

DATA="/kaggle/input/datasets/arpitjoshua/mt-trajnet-thesis-data/kaggle_upload"
with open(DATA+"/assembled_trajectories.pkl","rb") as fh:
    data=pickle.load(fh)
X_traj=data["X_traj"]
y_arr=data["y_arr"]
groups=data["groups"]

print("loaded:",len(X_traj),"batches | y",y_arr.shape,"| codes",len(np.unique(groups)))

device: cuda
gpu: Tesla T4
loaded: 1005 batches | y (1005, 4) | codes 25


## Data preparation

In [2]:
STRIDE=2
MAXLEN=6000
targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]

def prep_batch(traj_list,mean,std,maxlen=MAXLEN):
    out=[]
    for a in traj_list:
        a=a[::STRIDE]
        a=(a-mean)/std
        if len(a)<maxlen:
            pad=np.zeros((maxlen-len(a),a.shape[1]),dtype="float32")
            a=np.vstack([a,pad])
        else:
            a=a[:maxlen]
        out.append(a)
    arr=np.array(out,dtype="float32")
    return torch.tensor(arr).transpose(1,2)

print("prep ready, stride",STRIDE,"maxlen",MAXLEN)

prep ready, stride 2 maxlen 6000


## Model classes

The encoder and evidential head are identical to the multi-task model. The single-task model has one head instead of four, sharing nothing across targets.

In [3]:
class TCNBlock(nn.Module):
    def __init__(self,in_ch,out_ch,dilation,kernel=3,dropout=0.1):
        super().__init__()
        pad=(kernel-1)*dilation
        self.conv1=nn.Conv1d(in_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.conv2=nn.Conv1d(out_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.relu=nn.ReLU()
        self.drop=nn.Dropout(dropout)
        self.pad=pad
        self.down=nn.Conv1d(in_ch,out_ch,1) if in_ch!=out_ch else None
    def forward(self,x):
        res=x if self.down is None else self.down(x)
        out=self.conv1(x)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        out=self.conv2(out)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        return self.relu(out+res)

class TCNEncoder(nn.Module):
    def __init__(self,in_ch=10,hidden=128,dropout=0.1):
        super().__init__()
        self.b1=TCNBlock(in_ch,hidden,1,dropout=dropout)
        self.b2=TCNBlock(hidden,hidden,2,dropout=dropout)
        self.b3=TCNBlock(hidden,hidden,4,dropout=dropout)
        self.b4=TCNBlock(hidden,hidden,8,dropout=dropout)
    def forward(self,x):
        x=self.b1(x);x=self.b2(x);x=self.b3(x);x=self.b4(x)
        return x.mean(dim=2)

class EvidentialHead(nn.Module):
    def __init__(self,in_dim,hidden=64,dropout=0.1):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,hidden),nn.ReLU(),nn.Dropout(dropout),nn.Linear(hidden,4))
    def forward(self,x):
        out=self.net(x)
        gamma=out[:,0]
        nu=F.softplus(out[:,1]).clamp(min=1e-2,max=1e3)
        alpha=F.softplus(out[:,2]).clamp(min=1e-2,max=1e3)+1.0
        beta=F.softplus(out[:,3]).clamp(min=1e-2,max=1e3)
        return gamma,nu,alpha,beta

def evidential_loss(y,gamma,nu,alpha,beta,lam=1.0):
    om=2*beta*(1+nu)
    nll=(0.5*torch.log(np.pi/nu)-alpha*torch.log(om)
         +(alpha+0.5)*torch.log(nu*(y-gamma)**2+om)
         +torch.lgamma(alpha)-torch.lgamma(alpha+0.5))
    reg=torch.abs(y-gamma)*(2*nu+alpha)
    return (nll+lam*reg).mean()

class SingleTaskNet(nn.Module):
    def __init__(self,in_ch=10,hidden=128,dropout=0.1):
        super().__init__()
        self.encoder=TCNEncoder(in_ch,hidden,dropout)
        self.head=EvidentialHead(hidden,dropout=dropout)
    def forward(self,x):
        z=self.encoder(x)
        return self.head(z)

print("model classes ready")

model classes ready


## Stratified folds

The same stratified fold assignment as the authoritative run: code 15 held out, high-cluster codes balanced 22/6/6 across the three folds. Single-task models train and evaluate on exactly these folds.

In [4]:
OOD_CODE=15
fold_codes=[
[1,2,4,6,7,9,13,18,25],
[10,11,17,19,22,24],
[3,5,8,12,14,16,20,21,23],
]
n_splits=3

fold_test_idx=[]
for f in range(n_splits):
    idx=np.where(np.isin(groups,fold_codes[f]))[0]
    fold_test_idx.append(idx)

for f in range(n_splits):
    print(f"fold {f+1}: {len(fold_test_idx[f])} test batches")
print("total CV batches:",sum(len(x) for x in fold_test_idx),"(code 15 excluded)")

fold 1: 314 test batches
fold 2: 313 test batches
fold 3: 314 test batches
total CV batches: 941 (code 15 excluded)


## Single-task training

One model per target, each trained on the same stratified folds as the multi-task run, with the same seed and the same full-fold training (no calibration partition removed, matching the cross-fitting setup). A small validation split is used for early stopping only.

In [5]:
from sklearn.metrics import mean_squared_error
import time,warnings
warnings.filterwarnings("ignore")

def train_single(target_idx,max_epochs=150,lr=5e-4,batch_size=16,patience=10):
    fold_rmse=[]
    for f in range(n_splits):
        te=fold_test_idx[f]
        trv=np.concatenate([fold_test_idx[j] for j in range(n_splits) if j!=f])
        g=groups[trv];uniq=np.unique(g)
        rng=np.random.RandomState(SEED+f)
        val_codes=rng.choice(uniq,size=max(1,len(uniq)//8),replace=False)
        va=trv[np.isin(g,val_codes)]
        tr=trv[~np.isin(g,val_codes)]

        xmean=np.vstack([X_traj[i][::STRIDE] for i in tr]).mean(axis=0)
        xstd=np.vstack([X_traj[i][::STRIDE] for i in tr]).std(axis=0)+1e-6
        Xtr=prep_batch([X_traj[i] for i in tr],xmean,xstd)
        Xva=prep_batch([X_traj[i] for i in va],xmean,xstd)
        Xte=prep_batch([X_traj[i] for i in te],xmean,xstd)
        ymean=y_arr[tr,target_idx].mean();ystd=y_arr[tr,target_idx].std()+1e-6
        ytr=torch.tensor((y_arr[tr,target_idx]-ymean)/ystd,dtype=torch.float32)
        yva=torch.tensor((y_arr[va,target_idx]-ymean)/ystd,dtype=torch.float32)
        yte=y_arr[te,target_idx]

        torch.manual_seed(SEED+f)
        model=SingleTaskNet().to(device)
        opt=torch.optim.Adam(model.parameters(),lr=lr)
        best=float("inf");best_state=None;wait=0
        for ep in range(max_epochs):
            model.train()
            perm=torch.randperm(len(Xtr))
            for i in range(0,len(Xtr),batch_size):
                idx=perm[i:i+batch_size]
                xb=Xtr[idx].to(device);yb=ytr[idx].to(device)
                opt.zero_grad()
                gg,nn_,aa,bb=model(xb)
                loss=evidential_loss(yb,gg,nn_,aa,bb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                opt.step()
            model.eval()
            with torch.no_grad():
                vl=0
                for i in range(0,len(Xva),batch_size):
                    xb=Xva[i:i+batch_size].to(device);yb=yva[i:i+batch_size].to(device)
                    gg,nn_,aa,bb=model(xb)
                    vl+=evidential_loss(yb,gg,nn_,aa,bb).item()
            if vl<best-1e-4:
                best=vl;best_state={k:v.cpu().clone() for k,v in model.state_dict().items()};wait=0
            else:
                wait+=1
                if wait>=patience:break
        model.load_state_dict(best_state);model.eval()

        preds=[]
        with torch.no_grad():
            for i in range(0,len(Xte),batch_size):
                xb=Xte[i:i+batch_size].to(device)
                gg,_,_,_=model(xb)
                preds.append(gg.cpu().numpy())
        pred=np.concatenate(preds)*ystd+ymean
        fold_rmse.append(np.sqrt(mean_squared_error(yte,pred)))
    return np.array(fold_rmse)

print("ready")

ready


## Train one model per target

Four separate models on the stratified folds. Each prints its per-fold RMSE as it finishes.

In [6]:
t0=time.time()
single_folds={}
for j,t in enumerate(targets):
    print(f"training {t} ...")
    r=train_single(target_idx=j)
    single_folds[t]=[float(v) for v in r]
    print(f"  {t}: {r.mean():.3f} (std {r.std():.3f})  folds {[round(v,3) for v in r]}")
print("\ntime:",round(time.time()-t0),"seconds")

training dissolution_av ...
  dissolution_av: 3.547 (std 0.111)  folds [np.float64(3.654), np.float64(3.393), np.float64(3.593)]
training tbl_av_hardness ...
  tbl_av_hardness: 8.310 (std 1.996)  folds [np.float64(11.054), np.float64(7.514), np.float64(6.363)]
training tbl_rsd_weight ...
  tbl_rsd_weight: 0.595 (std 0.125)  folds [np.float64(0.768), np.float64(0.54), np.float64(0.477)]
training fct_tensile ...
  fct_tensile: 0.364 (std 0.038)  folds [np.float64(0.385), np.float64(0.311), np.float64(0.397)]

time: 1473 seconds


## RQ2: single-task against multi-task

I compare the per-fold RMSE of the single-task models against the multi-task fold values from the authoritative run in Notebook 8. Both use the same stratified folds, the same seed, and the same full-fold training, so the only difference is the shared encoder. With three folds the test has low power, so p-values are weak evidence in either direction.

In [7]:
from scipy import stats
import json

multi_folds={
"dissolution_av":[3.344,3.147,3.292],
"tbl_av_hardness":[7.900,7.428,7.767],
"tbl_rsd_weight":[0.770,0.442,0.543],
"fct_tensile":[0.404,0.350,0.283],
}
multi_mean_auth={"dissolution_av":3.261,"tbl_av_hardness":7.698,"tbl_rsd_weight":0.585,"fct_tensile":0.345}

print(f"{'target':<18}{'single':<10}{'multi':<10}{'diff':<10}{'p-value':<10}{'significant'}")
rq2={}
for t in targets:
    s=np.array(single_folds[t]);m=np.array(multi_folds[t])
    d=s-m
    n=len(d);mean_d=d.mean();var_d=d.var(ddof=1)
    cv=var_d*(1/n+0.5)
    tstat=mean_d/np.sqrt(cv) if cv>0 else 0
    p=2*(1-stats.t.cdf(abs(tstat),n-1))
    single_mean=round(float(s.mean()),3)
    disp_diff=round(single_mean-multi_mean_auth[t],3)
    rq2[t]={"single_folds":[round(v,3) for v in s],"single_mean":single_mean,
            "multi_folds":[float(x) for x in m],"multi_mean":multi_mean_auth[t],"diff":disp_diff,
            "p":round(float(p),3),"significant":bool(p<0.05)}
    print(f"{t:<18}{single_mean:<10.3f}{multi_mean_auth[t]:<10.3f}{disp_diff:<10.3f}{p:<10.3f}{'yes' if p<0.05 else 'no'}")

single_params=358852
multi_params=384400
with open("/kaggle/working/rq2_stratified.json","w") as fh:
    json.dump({"comparison":"single-task vs multi-task, stratified folds + cross-fitting setup (seed 42), Nadeau-Bengio",
               "note":"multi-task fold values and means are from the Notebook 8 authoritative run; both on identical stratified folds; three folds gives low power. Fold values are stored rounded to three decimals, so recomputing a mean or a p-value from them can differ from the stored value by up to 0.002; multi_mean is Notebook 8 mean from the unrounded folds.",
               "params":{"multi_task":multi_params,"four_single_task":4*single_params,"reduction":round(4*single_params/multi_params,2)},
               "results":rq2},fh,indent=2)
print("\nsaved rq2_stratified.json")

target            single    multi     diff      p-value   significant
dissolution_av    3.547     3.261     0.286     0.012     yes
tbl_av_hardness   8.310     7.698     0.612     0.800     no
tbl_rsd_weight    0.595     0.585     0.010     0.904     no
fct_tensile       0.364     0.346     0.019     0.830     no

saved rq2_stratified.json


### Correction to the multi-task tensile mean

The table printed by the cell above reports the multi-task tensile RMSE as 0.346. That value was recomputed from the fold list, which is stored rounded to three decimals, and 0.404, 0.350 and 0.283 average to 0.3457. The authoritative mean from Notebook 8, computed from the unrounded fold values, is 0.345. The same rounding is why the tensile row shows a diff of 0.019 while 0.364 minus 0.346 is 0.018.

The cell source and the saved rq2_stratified.json now take the multi-task means directly from Notebook 8, so tensile reads 0.345 and the diff column is consistent with the two means beside it. The other three targets are unaffected because their rounded folds average to the same value as the unrounded ones.

The cell output below is left as the run produced it. No p-value and no conclusion changes: tensile remains not significant at p 0.830, and multi-task remains significantly better on dissolution at p 0.012. The notebook was not re-run, because re-running would retrain the four single-task models on the GPU to correct a display value.

## Summary

RQ2 on the stratified folds. Multi-task fold values come from the authoritative Notebook 8 run.

In [8]:
print("RQ2 SUMMARY (stratified folds, seed 42)\n")
print(f"{'target':<18}{'single-task':<14}{'multi-task':<14}{'p-value':<10}{'better'}")
for t in targets:
    r=rq2[t]
    better="multi-task" if r["single_mean"]>r["multi_mean"] else "single-task"
    print(f"{t:<18}{r['single_mean']:<14}{r['multi_mean']:<14}{r['p']:<10}{better}")

print(f"\nparameters: multi-task {multi_params:,} | four single-task {4*single_params:,}")
print(f"reduction: {4*single_params/multi_params:.2f}x fewer parameters with the shared encoder")
print("\nmulti-task significantly better on dissolution (p=0.012); no significant difference on the other three, multi-task numerically better on all")

RQ2 SUMMARY (stratified folds, seed 42)

target            single-task   multi-task    p-value   better
dissolution_av    3.547         3.261         0.012     multi-task
tbl_av_hardness   8.31          7.698         0.8       multi-task
tbl_rsd_weight    0.595         0.585         0.904     multi-task
fct_tensile       0.364         0.346         0.83      multi-task

parameters: multi-task 384,400 | four single-task 1,435,408
reduction: 3.73x fewer parameters with the shared encoder

multi-task significantly better on dissolution (p=0.012); no significant difference on the other three, multi-task numerically better on all
